# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. This helps you know what structures the dataset defines and how the data is organized.

We'll extract and inspect all available record sets, fields, and columns with their full Croissant `@id` for precise references.

In [ ]:
# List all record sets and their fields using '@id'
print("Available Record Sets:")
for rs in dataset.metadata.record_sets:
    print(f"  RecordSet @id: {rs['@id']}")
    print(f"    name: {rs.get('name', None)}")
    # List Fields
    if 'fields' in rs:
        print("    Fields:")
        for field in rs['fields']:
            print(f"      Field @id: {field['@id']} - {field.get('name', None)} - dataType: {field.get('dataType', None)}")
            # List Columns associated with this field
            if 'column' in field:
                if isinstance(field['column'], list):
                    for col in field['column']:
                        if isinstance(col, dict):
                            print(f"        Column @id: {col.get('@id', col)}")
                        else:
                            print(f"        Column @id: {col}")
                else:
                    col = field['column']
                    if isinstance(col, dict):
                        print(f"        Column @id: {col.get('@id', col)}")
                    else:
                        print(f"        Column @id: {col}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id` from the overview above.

**Note:** You must use the `@id` for all accesses; update the `record_sets_to_load` list below based on the printed overview above.

In [ ]:
# Example: Load all available record sets
from collections.abc import Iterable

# Helper: Get all record set @id values
record_sets_to_load = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_to_load:
    print(f"\nLoading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded shape for {record_set_id}: {df.shape}")
        print(f"Columns in {record_set_id}: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for {record_set_id}")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

The operations below are generic - **adapt the `chosen_record_set_id`, `numeric_field_id`, and `group_field_id` below to your data by substituting the `@id` you wish to analyze based on your earlier overview and column review above.**

In [ ]:
# Replace these IDs with those found in your dataset above for demonstration purposes.
if len(dataframes) > 0:
    # Use the first record set and its first suitable numeric and grouping field as a demo
    chosen_record_set_id = list(dataframes.keys())[0]
    df = dataframes[chosen_record_set_id]
    print(f"Working on: {chosen_record_set_id}")
    
    # Find a numeric column: Try standard protein metrics names
    numeric_candidates = [col for col in df.columns if df[col].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_candidates:
        # Find via substring, e.g., look for "MW", "peptide", "abundance", "count", etc.
        numeric_candidates = [col for col in df.columns if any(key in col.lower() for key in ["mw", "weight", "abundance", "count", "coverage", "peptide"])]
    print(f"Numeric field candidates: {numeric_candidates}")
    
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        # Try to coerce to numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors="coerce")
        threshold = df[numeric_field].mean()  # Use mean as a threshold example
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Find a group/categorical field
        group_field_candidates = [col for col in df.columns if df[col].nunique() < df.shape[0] / 4 and df[col].dtype == object]
        print(f"Group candidates: {group_field_candidates}")
        group_field = group_field_candidates[0] if group_field_candidates else None
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric fields found to analyze in this record set.")
else:
    print("No dataframes loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The example below makes use of a numeric field and a group field, if possible. Modify field names as needed for your dataset!

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and not filtered_df.empty and 'numeric_field' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group if possible
    if 'group_field' in locals() and group_field is not None and group_field in filtered_df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Filtered dataframe or numeric field not available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook has demonstrated:
- How to load Croissant datasets using `mlcroissant` by referencing all entities with their `@id`.
- Reviewing the structure of all record sets, fields, and columns.
- Extracting and processing records for EDA.
- Performing basic statistical filtering, normalization, and grouping.
- Visualizing data distributions for initial scientific insights.

You are encouraged to adapt field and record set selections to fit your specific scientific questions and the precise organization of this or any Croissant-compliant dataset.